# Code to analyze glacier elevation change (dH) rasters for Nevados de Chillán 
### performing geostpatial statistical analysis 

In [37]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import richdem as rd
import os

# --------- USER INPUT ---------

# Path to the reference DEM (used to calculate slope, aspect, elevation)
dem_path = "/Users/milliespencer/Desktop/dems_20250721_updated/SRTM_projected_raster.tif"

# Glacier outline shapefiles
glacier_shapefiles = {
    "2000": "/Users/milliespencer/Glacier-DEM-coregistration-and-MB/example_data_Nevados/Nevados_glacier_shapefiles/Nevados_shapefile_DGA2000/Nevados_polygons_DGA2000.shp",
    "2019": "/Users/milliespencer/Glacier-DEM-coregistration-and-MB/example_data_Nevados/Nevados_glacier_shapefiles/Nevados_shapefile_DGA2019/Nevados_polygons_DGA2019.shp"
}

# Glacier ID field
glacier_id_field = "COD_GLA"

# dH rasters
dh_raster_paths = {
    "SRTMminusIGM_2000": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseIGM_poly2000_updated/SRTM_projected_raste_clp_diff.tif",
    "CBminusSRTM_2000": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseSRTM_poly2000/CerroBlanco_DEMproje_clp_diff.tif",
    "LTminusSRTM_2000": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseSRTM_poly2000/LasTermas_DEMproject_clp_diff.tif",
    "CBminusIGM_2000": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseIGM_poly2000_updated/CerroBlanco_DEMproje_clp_diff.tif",
    "LTminusIGM_2000": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseIGM_poly2000_updated/LasTermas_DEMproject_clp_diff.tif",
    "SRTMminusIGM_2019": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseIGM_poly2019_updated/SRTM_projected_raste_clp_diff.tif",
    "CBminusSRTM_2019": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseSRTM_poly2019/CerroBlanco_DEMproje_clp_diff.tif",
    "LTminusSRTM_2019": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseSRTM_poly2019/LasTermas_DEMproject_clp_diff.tif",
    "CBminusIGM_2019": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseIGM_poly2019_updated/CerroBlanco_DEMproje_clp_diff.tif",
    "LTminusIGM_2019": "/Users/milliespencer/Desktop/dems_20250721_updated/output_data_baseIGM_poly2019_updated/LasTermas_DEMproject_clp_diff.tif"
}


# --------- HELPER FUNCTIONS ---------

def calculate_terrain_attributes(dem_path):
    dem = rd.LoadGDAL(dem_path)
    slope = rd.TerrainAttribute(dem, attrib='slope_degrees')
    aspect = rd.TerrainAttribute(dem, attrib='aspect')
    elevation = dem
    return slope.astype(float), aspect.astype(float), elevation.astype(float)


def sample_glacier_pixels(dh_raster_path, slope_arr, aspect_arr, elev_arr, glacier_gdf, glacier_id_field):
    glacier_pixel_data = []

    with rasterio.open(dh_raster_path) as dh_src:
        dh_data = dh_src.read(1)
        dh_transform = dh_src.transform
        dh_crs = dh_src.crs

        # Ensure glacier geometries are in the same CRS as the dH raster
        if glacier_gdf.crs != dh_crs:
            glacier_gdf = glacier_gdf.to_crs(dh_crs)

        for _, row in glacier_gdf.iterrows():
            glacier_id = row[glacier_id_field]
            geom = [row.geometry]

            try:
                out_dh, out_transform = mask(dh_src, geom, crop=True, filled=True, nodata=np.nan)
                out_dh = out_dh[0]

                bounds = row.geometry.bounds
                row_start, col_start = ~dh_transform * (bounds[0], bounds[3])
                row_start, col_start = int(row_start), int(col_start)

                h, w = out_dh.shape
                slope_crop = slope_arr[row_start:row_start + h, col_start:col_start + w]
                aspect_crop = aspect_arr[row_start:row_start + h, col_start:col_start + w]
                elev_crop = elev_arr[row_start:row_start + h, col_start:col_start + w]

                valid_mask = ~np.isnan(out_dh) & ~np.isnan(slope_crop) & ~np.isnan(aspect_crop) & ~np.isnan(elev_crop)

                df = pd.DataFrame({
                    "glacier_id": glacier_id,
                    "dH": out_dh[valid_mask],
                    "slope": slope_crop[valid_mask],
                    "aspect": aspect_crop[valid_mask],
                    "elevation": elev_crop[valid_mask]
                })

                glacier_pixel_data.append(df)

            except Exception as e:
                print(f"⚠️  Skipping glacier {glacier_id} due to error: {e}")
                continue

    return pd.concat(glacier_pixel_data, ignore_index=True) if glacier_pixel_data else pd.DataFrame()


# --------- MAIN SCRIPT ---------

def main():
    print("🟢 Calculating slope, aspect, elevation from reference DEM...")
    slope_arr, aspect_arr, elev_arr = calculate_terrain_attributes(dem_path)

    for year_label, shapefile_path in glacier_shapefiles.items():
        print(f"\n📂 Processing glaciers from year: {year_label}")
        glacier_gdf = gpd.read_file(shapefile_path)

        for raster_label, dh_path in dh_raster_paths.items():
            print(f"🔵 Processing raster: {raster_label} with outlines from {year_label}")

            pixel_df = sample_glacier_pixels(
                dh_raster_path=dh_path,
                slope_arr=slope_arr,
                aspect_arr=aspect_arr,
                elev_arr=elev_arr,
                glacier_gdf=glacier_gdf,
                glacier_id_field=glacier_id_field
            )

            if pixel_df.empty:
                print(f"⚠️ No valid pixels for {raster_label} with outlines {year_label}. Skipping.")
                continue

            output_prefix = f"{raster_label}_glaciers{year_label}"

            # Save pixel-level data (optional, comment out if not needed)
            pixel_csv = f"{output_prefix}_pixel_dh_data.csv"
            pixel_df.to_csv(pixel_csv, index=False)
            print(f"✅ Saved pixel-level data to: {pixel_csv}")

            # Glacier-level summary
            summary = pixel_df.groupby('glacier_id').agg({
                'dH': ['mean', 'std', 'count'],
                'slope': 'mean',
                'aspect': 'mean',
                'elevation': 'mean'
            })
            summary.columns = ['_'.join(col) for col in summary.columns]
            summary.reset_index(inplace=True)

            summary_csv = f"{output_prefix}_summary.csv"
            summary.to_csv(summary_csv, index=False)
            print(f"📊 Saved summary stats to: {summary_csv}")


if __name__ == "__main__":
    main()


🟢 Calculating slope, aspect, elevation from reference DEM...



A Slope calculation (degrees)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.243498======================= ] (99% - 0.0s - 1 threads)

A Aspect attribute calculation
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.489387======================= ] (99% - 0.0s - 1 threads)



📂 Processing glaciers from year: 2000
🔵 Processing raster: SRTMminusIGM_2000 with outlines from 2000
✅ Saved pixel-level data to: SRTMminusIGM_2000_glaciers2000_pixel_dh_data.csv
📊 Saved summary stats to: SRTMminusIGM_2000_glaciers2000_summary.csv
🔵 Processing raster: CBminusSRTM_2000 with outlines from 2000
⚠️  Skipping glacier CL108130010 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108100003 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108100002 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108100004 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108130008 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108130003 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108130007 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL108130006 due to error: Input shapes do not overlap raster.
⚠️  Skipping glacier CL10

## Compute mean slope, aspect, and elevation of each glacier, analyze whether elevation change is correlated with geospatial variable

In [39]:
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# --------- USER SETTINGS ---------
csv_folder = "/Users/milliespencer/Glacier-DEM-coregistration-and-MB"
file_pattern = os.path.join(csv_folder, "*_summary.csv")
output_folder = os.path.join(csv_folder, "correlation_analysis")
os.makedirs(output_folder, exist_ok=True)

# --------- HELPER FUNCTIONS ---------

def calculate_and_plot_correlations(df, raster_label, output_folder):
    """
    For a given dataframe, compute and plot correlations between dH and terrain attributes.
    """
    results = []
    for var in ['elevation_mean', 'slope_mean', 'aspect_mean']:
        try:
            r, p = pearsonr(df['dH_mean'], df[var])
            results.append({
                "Raster": raster_label,
                "Variable": var,
                "Pearson_r": r,
                "p_value": p
            })

            sns.regplot(data=df, x=var, y='dH_mean', scatter_kws={'alpha': 0.5})
            plt.title(f"{raster_label}: dH_mean vs {var}\nPearson r = {r:.2f}, p = {p:.3f}")
            plt.xlabel(var)
            plt.ylabel("Mean dH (m)")
            plt.tight_layout()
            plot_path = os.path.join(output_folder, f"{raster_label}_{var}_correlation.png")
            plt.savefig(plot_path)
            plt.close()
            print(f"📈 Saved plot: {plot_path}")

        except Exception as e:
            print(f"⚠️ Could not calculate correlation for {raster_label} vs {var}: {e}")
    return results

def extract_years_from_label(label):
    """
    Extract raster year and glacier year from a filename label.
    Expects parts like '..._2019_glaciers2019_...' or '..._2000_glaciers2000_...'
    """
    parts = label.split('_')
    raster_year = None
    glacier_year = None
    
    for p in parts:
        if p in ['2000', '2019']:
            raster_year = p
        elif p.startswith('glaciers'):
            year_part = p.replace('glaciers', '')
            if year_part in ['2000', '2019']:
                glacier_year = year_part

    return raster_year, glacier_year

# --------- MAIN ANALYSIS ---------

def main():
    all_results = []

    for filepath in glob.glob(file_pattern):
        filename = os.path.basename(filepath)
        raster_label = filename.replace("_summary.csv", "")

        raster_year, glacier_year = extract_years_from_label(raster_label)

        if raster_year is None or glacier_year is None:
            print(f"⚠️ Cannot determine years for {filename}, skipping.")
            continue

        if raster_year != glacier_year:
            print(f"⚠️ Skipping {filename} because raster year ({raster_year}) != glacier year ({glacier_year})")
            continue

        print(f"\n🔍 Analyzing: {raster_label}")

        try:
            df = pd.read_csv(filepath)
            df = df.dropna(subset=['dH_mean', 'slope_mean', 'aspect_mean', 'elevation_mean'])
            if df.empty:
                print(f"⚠️ Skipped {raster_label} (no valid data)")
                continue

            res = calculate_and_plot_correlations(df, raster_label, output_folder)
            all_results.extend(res)

        except Exception as e:
            print(f"⚠️ Error reading {filename}: {e}")
            continue

    results_df = pd.DataFrame(all_results)
    results_csv = os.path.join(output_folder, "correlation_summary.csv")
    results_df.to_csv(results_csv, index=False)
    print(f"\n✅ Saved correlation summary table to: {results_csv}")


if __name__ == "__main__":
    main()



🔍 Analyzing: CBminusSRTM_2019_glaciers2019
⚠️ Could not calculate correlation for CBminusSRTM_2019_glaciers2019 vs elevation_mean: `x` and `y` must have length at least 2.
⚠️ Could not calculate correlation for CBminusSRTM_2019_glaciers2019 vs slope_mean: `x` and `y` must have length at least 2.
⚠️ Could not calculate correlation for CBminusSRTM_2019_glaciers2019 vs aspect_mean: `x` and `y` must have length at least 2.

🔍 Analyzing: CBminusIGM_2019_glaciers2019
⚠️ Could not calculate correlation for CBminusIGM_2019_glaciers2019 vs elevation_mean: `x` and `y` must have length at least 2.
⚠️ Could not calculate correlation for CBminusIGM_2019_glaciers2019 vs slope_mean: `x` and `y` must have length at least 2.
⚠️ Could not calculate correlation for CBminusIGM_2019_glaciers2019 vs aspect_mean: `x` and `y` must have length at least 2.
⚠️ Skipping SRTMminusIGM_2019_glaciers2000_summary.csv because raster year (2019) != glacier year (2000)

🔍 Analyzing: LTminusSRTM_2000_glaciers2000
📈 Saved

## Result: no statistically significant relationship between mean glacier elevation, aspect, or slope and mean glacier elevation loss. 

# What if we look at a per-pixel basis, rather than averaged across whole glaciers? 

In [40]:
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# --------- USER SETTINGS ---------
csv_folder = "/Users/milliespencer/Glacier-DEM-coregistration-and-MB"
file_pattern = os.path.join(csv_folder, "*_pixel_dh_data.csv")  # now pixel-level files
output_folder = os.path.join(csv_folder, "pixel_correlation_analysis")
os.makedirs(output_folder, exist_ok=True)

def calculate_and_plot_correlations(df, raster_label, output_folder):
    results = []
    for var in ['elevation', 'slope', 'aspect']:  # pixel columns (no _mean)
        try:
            r, p = pearsonr(df['dH'], df[var])
            results.append({
                "Raster": raster_label,
                "Variable": var,
                "Pearson_r": r,
                "p_value": p
            })

            sns.regplot(data=df, x=var, y='dH', scatter_kws={'alpha': 0.05, 's': 10})
            plt.title(f"{raster_label}: dH vs {var}\nPearson r = {r:.2f}, p = {p:.3f}")
            plt.xlabel(var)
            plt.ylabel("dH (m)")
            plt.tight_layout()
            plot_path = os.path.join(output_folder, f"{raster_label}_{var}_pixel_correlation.png")
            plt.savefig(plot_path)
            plt.close()
            print(f"📈 Saved plot: {plot_path}")

        except Exception as e:
            print(f"⚠️ Could not calculate correlation for {raster_label} vs {var}: {e}")
    return results

def extract_years_from_label(label):
    parts = label.split('_')
    raster_year = None
    glacier_year = None
    
    for p in parts:
        if p in ['2000', '2019']:
            raster_year = p
        elif p.startswith('glaciers'):
            year_part = p.replace('glaciers', '')
            if year_part in ['2000', '2019']:
                glacier_year = year_part

    return raster_year, glacier_year

def main():
    all_results = []

    for filepath in glob.glob(file_pattern):
        filename = os.path.basename(filepath)
        raster_label = filename.replace("_pixel_dh_data.csv", "")

        raster_year, glacier_year = extract_years_from_label(raster_label)

        if raster_year is None or glacier_year is None:
            print(f"⚠️ Cannot determine years for {filename}, skipping.")
            continue

        if raster_year != glacier_year:
            print(f"⚠️ Skipping {filename} because raster year ({raster_year}) != glacier year ({glacier_year})")
            continue

        print(f"\n🔍 Analyzing pixels in: {raster_label}")

        try:
            df = pd.read_csv(filepath)
            df = df.dropna(subset=['dH', 'slope', 'aspect', 'elevation'])
            if df.empty:
                print(f"⚠️ Skipped {raster_label} (no valid pixel data)")
                continue

            res = calculate_and_plot_correlations(df, raster_label, output_folder)
            all_results.extend(res)

        except Exception as e:
            print(f"⚠️ Error reading {filename}: {e}")
            continue

    results_df = pd.DataFrame(all_results)
    results_csv = os.path.join(output_folder, "pixel_correlation_summary.csv")
    results_df.to_csv(results_csv, index=False)
    print(f"\n✅ Saved pixel-level correlation summary table to: {results_csv}")

if __name__ == "__main__":
    main()



🔍 Analyzing pixels in: SRTMminusIGM_2019_glaciers2019
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/SRTMminusIGM_2019_glaciers2019_elevation_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/SRTMminusIGM_2019_glaciers2019_slope_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/SRTMminusIGM_2019_glaciers2019_aspect_pixel_correlation.png

🔍 Analyzing pixels in: SRTMminusIGM_2000_glaciers2000
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/SRTMminusIGM_2000_glaciers2000_elevation_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/SRTMminusIGM_2000_glaciers2000_slope_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/SRTMminusI

/var/folders/3j/6dy_9gxj7vvgct178jkkp1680000gn/T/ipykernel_63466/2264903252.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(df['dH'], df[var])


📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/CBminusSRTM_2019_glaciers2019_aspect_pixel_correlation.png
⚠️ Skipping SRTMminusIGM_2019_glaciers2000_pixel_dh_data.csv because raster year (2019) != glacier year (2000)
⚠️ Skipping SRTMminusIGM_2000_glaciers2019_pixel_dh_data.csv because raster year (2000) != glacier year (2019)
⚠️ Cannot determine years for FullComplex_pixel_dh_data.csv, skipping.
⚠️ Skipping CBminusIGM_2000_glaciers2019_pixel_dh_data.csv because raster year (2000) != glacier year (2019)

🔍 Analyzing pixels in: CBminusIGM_2019_glaciers2019
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/CBminusIGM_2019_glaciers2019_elevation_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/CBminusIGM_2019_glaciers2019_slope_pixel_correlation.png


/var/folders/3j/6dy_9gxj7vvgct178jkkp1680000gn/T/ipykernel_63466/2264903252.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(df['dH'], df[var])


📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/CBminusIGM_2019_glaciers2019_aspect_pixel_correlation.png

🔍 Analyzing pixels in: LTminusSRTM_2000_glaciers2000
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/LTminusSRTM_2000_glaciers2000_elevation_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/LTminusSRTM_2000_glaciers2000_slope_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/LTminusSRTM_2000_glaciers2000_aspect_pixel_correlation.png

🔍 Analyzing pixels in: LTminusSRTM_2019_glaciers2019
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/LTminusSRTM_2019_glaciers2019_elevation_pixel_correlation.png
📈 Saved plot: /Users/milliespencer/Glacier-DEM-coregistration-and-MB/pixel_correlation_analysis/LTminusSRTM_2019_

In [41]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import pearsonr

# Enable inline plotting
%matplotlib inline

# Folder with the pixel-level merged CSVs
csv_folder = "/Users/milliespencer/Glacier-DEM-coregistration-and-MB/merged_csvs_pixels/"

# List CSVs in folder
csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]

# Loop through each CSV
for csv_file in csv_files:
    csv_path = os.path.join(csv_folder, csv_file)
    print(f"\n📂 Analyzing: {csv_file}")
    
    # Load data
    df = pd.read_csv(csv_path)

    # Remove any rows with missing or invalid values
    df_clean = df.dropna(subset=['dh', 'elevation', 'slope', 'aspect'])

    # Print basic stats
    print(f"✅ Number of valid pixels: {len(df_clean)}")

    # Compute Pearson correlation coefficients
    corr_dh_elev, _ = pearsonr(df_clean['elevation'], df_clean['dh'])
    corr_dh_slope, _ = pearsonr(df_clean['slope'], df_clean['dh'])
    corr_dh_aspect, _ = pearsonr(df_clean['aspect'], df_clean['dh'])

    print(f"📊 Correlation between dH and Elevation: {corr_dh_elev:.2f}")
    print(f"📊 Correlation between dH and Slope: {corr_dh_slope:.2f}")
    print(f"📊 Correlation between dH and Aspect: {corr_dh_aspect:.2f}")

    # Scatter plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.scatterplot(x='elevation', y='dh', data=df_clean, ax=axes[0], s=10)
    axes[0].set_title(f'dH vs Elevation\nr = {corr_dh_elev:.2f}')
    axes[0].set_xlabel('Elevation (m)')
    axes[0].set_ylabel('Elevation Change (dH, m)')

    sns.scatterplot(x='slope', y='dh', data=df_clean, ax=axes[1], s=10)
    axes[1].set_title(f'dH vs Slope\nr = {corr_dh_slope:.2f}')
    axes[1].set_xlabel('Slope (degrees)')
    axes[1].set_ylabel('dH (m)')

    sns.scatterplot(x='aspect', y='dh', data=df_clean, ax=axes[2], s=10)
    axes[2].set_title(f'dH vs Aspect\nr = {corr_dh_aspect:.2f}')
    axes[2].set_xlabel('Aspect (degrees)')
    axes[2].set_ylabel('dH (m)')

    plt.tight_layout()
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/milliespencer/Glacier-DEM-coregistration-and-MB/merged_csvs_pixels/'